# Explore Delta Lake

In this notebook, you'll explore how to use Delta Lake in an Azure Databricks Spark cluster.

Your workspace uses **Unity Catalog**. Instead of the Databricks file system (DBFS) root, you'll store files in a Unity Catalog **volume** named **labdata** in the **default** schema of your workspace catalog.

## Ingest data

Use the **&#9656; Run Cell** menu option at the top-right of the following cell to run it. The cell creates the **labdata** volume, removes any files and tables left over from a previous run, and downloads a data file into the volume.

In [ ]:
import urllib.request

# The workspace catalog (named after your Azure Databricks workspace)
catalog = spark.sql("SELECT current_catalog()").first()[0]
volume_path = f"/Volumes/{catalog}/default/labdata"

# Create the volume, and clean up anything left over from a previous run
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.default.labdata")
spark.sql("DROP SCHEMA IF EXISTS adventureworks CASCADE")
for folder in ["delta", "device_stream"]:
    dbutils.fs.rm(f"{volume_path}/{folder}", True)

# Download the product data into the volume
urllib.request.urlretrieve("https://raw.githubusercontent.com/CloudLabs-MOC/dp-203-azure-data-engineer/guidedlabs-Azure-Databricks/Allfiles/labs/25/data/products.csv", f"{volume_path}/products.csv")

print("Catalog:", catalog)
print("Volume path:", volume_path)
display(dbutils.fs.ls(volume_path))

Now that you've ingested the data, you can load it into a Spark dataframe from the volume.

In [ ]:
df = spark.read.load(f"{volume_path}/products.csv", format="csv", header=True)
display(df.limit(10))

## Load the file data into a delta table

You can persist the data in the dataframe in Delta format using the following code:

In [ ]:
delta_table_path = f"{volume_path}/delta/products-delta"
df.write.format("delta").save(delta_table_path)

The data for a delta lake table is stored in Parquet format. A log file is also created to track modifications made to the data.

Run the following cell to view the contents of the folder where the delta data has been saved. Note the Parquet data file and the **_delta_log** folder.

In [ ]:
display(dbutils.fs.ls(delta_table_path))

The file data in Delta format can be loaded into a **DeltaTable** object, which you can use to view and update the data in the table. Run the following cell to update the data; reducing the price of product 771 by 10%.

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)

# Update the table (reduce price of product 771 by 10%)
deltaTable.update(
    condition = "ProductID == 771",
    set = { "ListPrice": "ListPrice * 0.9" })

# View the updated data as a dataframe
deltaTable.toDF().filter("ProductID == 771").show()

The update is persisted to the data in the delta folder, and will be reflected in any new dataframe loaded from that location:

In [ ]:
new_df = spark.read.format("delta").load(delta_table_path)
new_df.filter("ProductID==771").show()

Data modifications are logged, enabling you to use the *time-travel* capabilities of Delta Lake to view previous versions of the data. For example, use the following code to view the original version of the product data:

In [ ]:
new_df = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
new_df.show(10)

The log contains a full history of modifications to the data. Use the following code to see a record of the last 10 changes:

In [ ]:
deltaTable.history(10).show(10, False, True)

## Create catalog tables

So far you've worked with delta data by loading it from the folder containing the Parquet files on which it is based. You can also define *catalog tables* that encapsulate the data and provide a named table entity that you can reference in SQL code.

Unity Catalog supports two kinds of tables:

- *Managed* tables, whose data files are stored and managed by Unity Catalog in the storage for your catalog.
- *External* tables, whose data files are stored in a cloud storage location that an administrator has registered as an *external location*.

> **Note**: This lab environment doesn't include an external location, so you can't create an external table. You can still query Delta files in a volume directly by their path, as you'll do now.

### Query Delta files by path

Run the following cell to query the Delta files in the volume with SQL, without defining a table.

In [ ]:
display(spark.sql(f"SELECT * FROM delta.`{delta_table_path}`"))

### Create a managed table

Run the following code to create a new schema named **AdventureWorks**, and then create (and describe) a managed table named **ProductsManaged** based on the dataframe you originally loaded from the **products.csv** file (before you updated the price of product 771).

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS AdventureWorks")
df.write.format("delta").saveAsTable("AdventureWorks.ProductsManaged")
spark.sql("DESCRIBE EXTENDED AdventureWorks.ProductsManaged").show(truncate=False)

You did not specify a location for the Parquet files used by the table. Unity Catalog manages them for you, and the **Type** property is **MANAGED**. The **Location** property shows the managed storage for your catalog (an **abfss://** path in Azure Data Lake Storage).

You can query the table by name with a SQL `SELECT` statement:

In [ ]:
%sql
USE AdventureWorks;

SELECT * FROM ProductsManaged;

### Compare managed tables and Delta files

Let's explore the difference between a managed table and the Delta files in your volume.

First, use the following code to list the tables in the **AdventureWorks** schema:

In [ ]:
%sql
USE AdventureWorks;

SHOW TABLES;

What happens if we use a `DROP` statement to delete the managed table?

In [ ]:
%sql
USE AdventureWorks;

DROP TABLE IF EXISTS ProductsManaged;

SHOW TABLES;

The managed table has been removed, and Unity Catalog deletes its data files for you. (Unity Catalog keeps the files for a short period, so a dropped managed table can be recovered with `UNDROP TABLE`.)

The Delta files in your volume aren't affected, because no table owns them. Run the following cell to confirm that they're still there:

In [ ]:
display(dbutils.fs.ls(delta_table_path))

You can use the following code to create a new managed table named **Products** from the Delta files in the **products-delta** folder:

In [ ]:
spark.sql(f"CREATE TABLE AdventureWorks.Products AS SELECT * FROM delta.`{delta_table_path}`")

Now you can query the new table:

In [ ]:
%sql
USE AdventureWorks;

SELECT * FROM Products;

Because the table was created from the current version of the Delta files, it reflects the modification you previously made to the price of product 771.

## Use delta tables for streaming data

Delta lake supports streaming data. Delta tables can be a *sink* or a *source* for data streams created using the Spark Structured Streaming API. In this example, you'll use a delta table as a sink for some streaming data in a simulated internet of things (IoT) scenario.

First, let's get some simulated device data in JSON format. Run the following cell to download a JSON file that looks like this:

```json
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
```

In [ ]:
import os

device_stream_path = f"{volume_path}/device_stream"
os.makedirs(device_stream_path, exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/CloudLabs-MOC/dp-203-azure-data-engineer/guidedlabs-Azure-Databricks/Allfiles/labs/25/data/devices1.json", f"{device_stream_path}/devices1.json")
display(dbutils.fs.ls(device_stream_path))

Now you're ready to use Spark Structured Streaming to create a stream based on the folder containing the JSON device data.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Create a stream that reads data from the folder, using a JSON schema
jsonSchema = StructType([
    StructField("device", StringType(), False),
    StructField("status", StringType(), False)
])
iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTrigger", 1).json(device_stream_path)
print("Source stream created...")

Now you'll take the stream of data you're reading from the folder, and perpetually write it to a managed delta table named **IotDeviceData**. The stream's checkpoint, which tracks the files it has already processed, is stored in the volume.

In [ ]:
# Write the stream to a delta table
checkpointpath = f"{volume_path}/delta/checkpoint"
deltastream = iotstream.writeStream.option("checkpointLocation", checkpointpath).toTable("AdventureWorks.IotDeviceData")
print("Streaming to delta sink...")

> **Note**: The stream keeps running in the background until you stop it, so the previous cell continues to show **Interrupt** and a stream status panel. This is expected. Don't interrupt it; continue with the next cell.

Wait a few seconds for the stream to process the first file. Then you can query the table just like any other:

In [ ]:
%sql
SELECT * FROM AdventureWorks.IotDeviceData;

Now let's add some fresh device data to the stream.

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/CloudLabs-MOC/dp-203-azure-data-engineer/guidedlabs-Azure-Databricks/Allfiles/labs/25/data/devices2.json", f"{device_stream_path}/devices2.json")
print("devices2.json added to the stream folder")

The new JSON data in the device folder is read into the stream and written to the delta table. Wait a few seconds, and then run the following cell to see that the table now contains the new rows (16 rows instead of 9):

In [ ]:
%sql
SELECT * FROM AdventureWorks.IotDeviceData;

To stop the stream, use its **stop** method:

In [ ]:
deltastream.stop()